# 🎛️ 04 - Interactive Executive Dashboard & Hype Inspector

### AI-Powered BS & Hype Analyzer — Competition Demo Notebook

This interactive dashboard allows decision-makers, financial analysts, and researchers to:
- **Filter in real-time** by media outlet, sector, and minimum hype threshold.
- **Inspect key performance indicators (KPIs)** on information pollution.
- **Search by ticker or topic** (`NVIDIA`, `Bitcoin`, `Fed`, `AI`).
- **Drill down into individual articles** with a 5-factor linguistic radar breakdown.
- **Explore the live narrative echo-chamber network**.

---

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.config import PROCESSED_DATA_DIR
from src.graph import EchoChamberGraphBuilder
from src.viz import (
    plot_hype_distribution,
    plot_outlet_comparison,
    plot_subjectivity_vs_hype,
    plot_echo_chamber_network_plotly,
    plot_hype_dimension_radar
)

# 1. Load Precomputed Features
df = pd.read_parquet(PROCESSED_DATA_DIR / 'sample_features.parquet')
builder = EchoChamberGraphBuilder()
G = builder.build_outlet_graph(df)
print(f'✓ Dashboard initialized with {len(df)} articles across {df["outlet"].nunique()} outlets!')

## 2. Live Executive Control Panel & KPI Cards

Use the interactive controls below to filter data and trigger live analytical updates.

In [ ]:
# UI Controls
outlet_options = ['All'] + sorted(list(df['outlet'].unique()))
category_options = ['All'] + sorted(list(df['category'].dropna().unique()))

outlet_dropdown = widgets.Dropdown(options=outlet_options, value='All', description='Outlet:', layout=widgets.Layout(width='300px'))
cat_dropdown = widgets.Dropdown(options=category_options, value='All', description='Category:', layout=widgets.Layout(width='300px'))
threshold_slider = widgets.FloatSlider(value=0.50, min=0.10, max=0.90, step=0.05, description='Hype Thresh:', continuous_update=False, layout=widgets.Layout(width='350px'))
search_box = widgets.Text(value='', placeholder='Search ticker, keyword (e.g. Nvidia, Bitcoin)...', description='Search:', layout=widgets.Layout(width='350px'))

controls_row1 = widgets.HBox([outlet_dropdown, cat_dropdown])
controls_row2 = widgets.HBox([threshold_slider, search_box])

out_kpis = widgets.Output()
out_charts = widgets.Output()
out_drilldown = widgets.Output()

def update_dashboard(*args):
    sub_df = df.copy()
    
    # Filter by outlet
    if outlet_dropdown.value != 'All':
        sub_df = sub_df[sub_df['outlet'] == outlet_dropdown.value]
        
    # Filter by category
    if cat_dropdown.value != 'All':
        sub_df = sub_df[sub_df['category'] == cat_dropdown.value]
        
    # Filter by search keyword
    q = search_box.value.strip().lower()
    if q:
        sub_df = sub_df[sub_df['title'].str.lower().str.contains(q) | sub_df['summary'].str.lower().str.contains(q)]
        
    thresh = threshold_slider.value
    flagged = sub_df[sub_df['hype_score'] >= thresh]
    flagged_pct = (len(flagged) / len(sub_df) * 100) if len(sub_df) > 0 else 0.0
    avg_hype = sub_df['hype_score'].mean() if len(sub_df) > 0 else 0.0
    
    with out_kpis:
        clear_output(wait=True)
        kpi_html = f'''
        <div style="display: flex; gap: 15px; margin: 15px 0;">
            <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 8px; border-left: 5px solid #00CC96;">
                <div style="color: #aaa; font-size: 12px;">TOTAL ANALYZED</div>
                <div style="font-size: 24px; font-weight: bold; color: #fff;">{len(sub_df)}</div>
            </div>
            <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 8px; border-left: 5px solid #FF6692;">
                <div style="color: #aaa; font-size: 12px;">HIGH-HYPE FLAGGED</div>
                <div style="font-size: 24px; font-weight: bold; color: #FF6692;">{len(flagged)} ({flagged_pct:.1f}%)</div>
            </div>
            <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 8px; border-left: 5px solid #FFA15A;">
                <div style="color: #aaa; font-size: 12px;">MEAN HYPE SCORE</div>
                <div style="font-size: 24px; font-weight: bold; color: #fff;">{avg_hype:.3f}</div>
            </div>
            <div style="flex: 1; background: #1e1e2e; padding: 15px; border-radius: 8px; border-left: 5px solid #636EFA;">
                <div style="color: #aaa; font-size: 12px;">ECHO GRAPH NODES</div>
                <div style="font-size: 24px; font-weight: bold; color: #fff;">{len(G.nodes)}</div>
            </div>
        </div>
        '''
        display(HTML(kpi_html))
        
    with out_charts:
        clear_output(wait=True)
        if sub_df.empty:
            display(HTML('<p style="color: #ff6666;">No items match the current filter criteria.</p>'))
        else:
            fig1 = plot_hype_distribution(sub_df, threshold=thresh)
            fig1.show()
            fig2 = plot_subjectivity_vs_hype(sub_df)
            fig2.show()

    with out_drilldown:
        clear_output(wait=True)
        if not sub_df.empty:
            top_item = sub_df.sort_values(by='hype_score', ascending=False).iloc[0].to_dict()
            radar_fig = plot_hype_dimension_radar(top_item)
            radar_fig.show()
            
            drill_html = f'''
            <div style="background: #181824; padding: 15px; border-radius: 8px; margin-top: 10px;">
                <h4 style="color: #FF6692; margin-top: 0;">⚠️ Highest Hype Item in Selected Scope</h4>
                <p><b>Title:</b> {top_item.get('title')}</p>
                <p><b>Outlet:</b> {top_item.get('outlet')} | <b>Category:</b> {top_item.get('category')} | <b>Hype Score:</b> {top_item.get('hype_score'):.3f}</p>
                <p><b>Summary:</b> {top_item.get('summary')}</p>
            </div>
            '''
            display(HTML(drill_html))

outlet_dropdown.observe(update_dashboard, names='value')
cat_dropdown.observe(update_dashboard, names='value')
threshold_slider.observe(update_dashboard, names='value')
search_box.observe(update_dashboard, names='value')

# Render Dashboard
display(widgets.VBox([controls_row1, controls_row2, out_kpis, out_charts, out_drilldown]))
update_dashboard()

## 3. Narrative Echo-Chamber Graph Exploration

Inspect the interactive network showing cross-outlet echo loops.

In [ ]:
fig_net = plot_echo_chamber_network_plotly(G)
fig_net.show()

## 4. Key Findings, Business Narrative & Decision Context

### Executive Summary
1. **Bimodal Information Landscape**: Financial media exhibits a sharp bimodal distribution. Institutional wire services (Reuters, WSJ, Bloomberg) cluster at hype scores below 0.25, while retail trading bullet points and crypto social channels spike above 0.75.
2. **Echo Chamber Transmission**: High-hype narratives consistently originate in social/video channels before being amplified by retail financial broadcasters (CNBC, MarketWatch), which act as transmission bridges with the highest network PageRank and betweenness centrality.
3. **Decision Impact**: Using the configurable threshold $\tau=0.50$, quantitative hedge funds and corporate risk analysts can programmatically filter noise, discount sensational rumors, and prevent FOMO-driven algorithmic trades.